In [2]:
from datetime import datetime as dt
from datetime import timedelta
import requests
import json
import pandas as pd
import polars as pl

In [4]:
now = dt.now().strftime("%Y-%m-%d")
search_url = "https://workspace.refinitiv.com/api/tm3-backend/muni-data-analysis/deal-search/search"
detail_url = 'https://workspace.refinitiv.com/api/tm3-backend/muni-data-analysis/common/deal-analysis?dealId={deal_id}&evaluationDate='+now
# cookie = """hmds-token=AQIC5wM2LY4SfczFtDZhmB5KBm4uGSsAl9aI8wcUZ75rBCM%3D%40AAJTSQACNDAAAlNLABM2MzQ2NDM3OTYzODM4MDg0MjI3AAJTMQACMzc%3D%23; x-login-domain=sts.identity.ciam.refinitiv.net; sessionId=5450653b-eaec-4ebd-873c-0059d214cedd; sharedSessionStartTime=1752615752814; userId=SL1-F2VMEAU; x-cdn-account=eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjhDQkMtSFMyNTYiLCJraWQiOiJmZmM3NDk4YzI2YmM0M2UyODFiMGE4NmZjZmQ4MDc1MiIsInppcCI6IkRFRiIsInR5cCI6IkpXVCIsImN0eSI6IkpXVCJ9.nAjtKFu9Vjro4--p1jqO1mx-acH2lb4kYesyQ6uPqVOmGogh7UtVMWHv3cihhi-3TTOXoxSW5oFMKikSG88-l8ujP1dhAOmT4LRpb8bZdI2tqySzZXF6_KI8Hc_JSz9VBEPM27j0EGK1hmw-hEMt42TtweUwUU7Rm9hflZbA0BroSTTcivFiG0jpsrA1Da0WlAmCWioi-HliJxNOKnxUdlaykYyxYmvgHWOaAKsJN5rbP_iap--5nw_Fj1f0vXUsegwU5VM8nVeFFOO2PVhHUUecEanFhz73eCwPPmYXb7JA7GCFukiwCWvmjQwFvch0KfW0Srp1G_PVc4-ZBiTpxQ.QXiJ5k18KBGF3VTr878U9w.dQdMSjm5g8cmanLIOiYW2wIrZ1aao1rM_dT7gMn9yO7zN6oB8fEllb-KNpRMGiusr73OizUskTX_edoVCPVwaq4uS8j7UCoYLs-4jr8pJ3Jx1FySds4eUrbgYS-Sd4qvfXm9KbH7zQXopOcyoWr7bn44qprbdeaAlqVkdzliLzRlj44s7rr7tNT_xY35B978jFF8FryMDN46T8s2xMZwT5fqOoGko3JrqmvbGFOdnR2MDnaqdhoaV7vJ_XJmM8YV5gk4wkNnYjOaW4BZx_PULrE0dQE6xbDft4teh5IbyIVL-ue5Dj1fu7FvhWTGm9I7Q4I3Dl3qr8EL362nDwIp5iulx4rlqf1Tfo7H8brxI7aUre0Za9RZtJ-vzZx5bqAgd1n2dx4YP1UFOTYkfhkJIPzZM0iWj04pNBWri2IsytQbnBbkKVXlqVfkbjuImjp7CLkORbA5wfsWWwa1Nvw2hNzlkl_IcoGx36Bj9TrJqVAfae3rUZUKWXwnhM__oDkBE8mOgiYUpRI9WcDSzXhqEWPj9UQma7-Vf8nd8Xt5mLXXgVkxNb3be5iqnbkjOuaJuT1Gs8LreZjlzMnmf6YdJgIT7YyXbwRoyMP-DAhCP8igbJ5NLV_LZ3qXe5cpAQzWtYD5e1ljaLxSrJRXFIvjwSaFZQuO2kJgAPHee_kyQmKJSbIBCtOF5btqUjm5jroSgkKYC1aB9z9sGekiDUNx-p3mwLlJo67L2gycy7IL2BPi-Z0h6eTHdEHmlV5QFi783qq16TJ-BTGfsIVEXodc7Fih2gmaNcn688lCYweOs-TuX49barraqG7rCS64r8YxKumrKsgob8x3i2-hDaf4Wm9R_2QxYwtQJumSYYP2rSJqBDgkxzO3M9tV5IQUm5PcfIunK3Nn6VNoHor_Neh_LqkOJuE0hsMaaP-9w0FrlsplnGTW6jZL-EXH_DWpk3abIyiNLBUX7Kp-DIQOeNxpBi7IjYcK8uq-ay2jgUOQoRNhsfj-I3GUZdf04ZtBwPEwpHL9YOrXS6kvDy4MvQwi3T4kJEPpH9x4zs4b2V1CiEdC-ym0BPYFWKttha4XgJ8lDIG4YLFClei6YcFeDqKLembeqiMr3a8Lzq64XlxPbe1ZTJ5gqaSZWt8yjuNjktqLYSvWY1dnRqDTgosYCNS3xfBG1OUSVR5y1Sq9zh_E9cDfEntnFOjoMkbNl5q_onpmMcqtgSoIw8eF33KvTd2emeOkEuz2EdIFtX801jyYWK9XNzEE67K3i1yeVBGrEldyqaoOvpsQao-9gB5I976s-zYxakft0lMpIP-N-caqn66BuVXdtVU3pIt7QkzEjZnYj5CZLMfGI7lcQSwwap6in7Ukza37KP0pnmVAda0Qmi1q44wdDNZTsJpejiiEYosQgYZPVVRD3LP77adMo5NpRkZ1b2E3QwOrGgVmhBQ7LShfVFzynia3UFrrHb_bu0ZRbo7lAUwEQLEyT6G7uIDqR9ZDh7nhTGVQjDYdnq8viyHu89Kh81Kilv4VAiAhAIswtIGvKvl5EZkOfcn_Ta5oFqhTkf0hhYPazetRLNP8gIOSd77UNxkm8_boHInOOIKEJ9Rp6EYkJ5kwdD-fDpIY23zWgzoh6sTmmcKLS6J4PnKAfSnnoMuio_pAe4JZlOgpF2cLC8lrPlpnin2gIAyUKK81jSFJJOdjk5Y9awCyQ-TISrvGtWeaIiNF3wXwWl3D.2vlaHVDPil3LhN8CsHSo4g; x-sts-token=eyJhbGciOiJSUzI1NiIsImtpZCI6ImRMdFd2Q0tCSC1NclVyWm9YMXFod2pZQ2t1eDV0V2ZSS2o4ME9vcjdUY28iLCJ0eXAiOiJhdCtqd3QifQ.eyJhdWQiOiIzNDYzYTU5MjEzNDQ0NDI4YmVlNWVhZDg4ZmYxZDkwMDQ0NDFiNjU0IiwiZGF0YSI6IntcImNpcGhlcnRleHRcIjpcIndKNVh5bVE5WU5tXy1aSWtUZGJJam43VmRXd3dGN2lCU3pfamxxVnhBZF91M21uX3BGeVk3cm0zUWVtWFdoUnhTTGNRSmlwNVRBTlc2WGpncHpDam8zYVBWekRfbkNHRG9tQjVOQkZBOFZLckVScWdLRnNHU1p0V2E5SWlDal9CRC00Y0RzeHREMDhBYWNHbFg3MXJMb3VBLVRldG9vczlXTXkyVFFraER5dDg5NE1SZkxRRWtBdFJhQWUwVmJDaFIzZ09SakdNeUNsOW90Snk1TzdNVzNJV1pDczZfakpiZUpLbEZwUE54clpOckNpTzdKdERqOUl0NnhrcnhMU2k4OFVmNVlyMHhidUV0ZTNaVnE1aGI1V2xTNE9qQk9RdWZtaFBUd0lLWU40dEh1dDk5bVd2MkpON3F3enZkU3Y4a0dhZDU4WEg1QXFTbXQwZi15TXotRl93MFFLVmV5Y0FyZXBHdXNMZ0hnbVlETW8yMHByYkp0M2dIeG9JbmJ1ajlGYnM2SGJfWndhUXhOV1FYNmh6ZFZtUUhwMXlFUGJsTG5Od1BGMWhvYzNQeVpfdHh3aG5ENzczOUlUV204LTNvdFVjbjF6ckVxdGpQSnplbW1zcU90UVJYdWVkcE5QRkZaTjE2d2duRzNqOW8wSFhFMERTOEg1LURCM0Jhc0Vib3VkbGtHdHlxUjN4M2MwT1RiaG9nYnY1Ni1RbHNJSnpadzJFOXlVYjEySF9nY3dXSTh4cHpldkxBb1FvN3ExRDRxbV9MQWJrdEFIdGlIdWdCc2NaZ01VUFl6em5POWJfLWt1YTdPSGpQVW05NlpBN0RMc3BIY0VaaG5Mck5GNjdKOWozdDJsSXluSGNjZ3pJam5qZTg1N2RsNDA2azhXa1Nna1JNa2FYMXRJMXhBeVhra09VbjhfeGVaaEVBZ3NtT3FkRjZnXCIsXCJlbmNyeXB0ZWRfa2V5XCI6XCJBUUlCQUhnNDJxMmVwUnBlNFpJWmQ1OHR1YUJFT3FWLTBOLXpmaVpmYUk1UXdTREJvQUhNTUpFanNNLWx4dWxSZGhMS0o3T0FBQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1Jb1JoeW5BY01nclVZM3V4QWdFUWdEdlF0RFA5d2dKWHIzYW5HV0owdUpmbUJJNkd5LXZMLUxLYjU3VEdCV2gzVEctRnRpQ3ZoYjFOaml1U0xBLVJPaFo1TVhvMTVYRXlFeXU1cmdcIixcImhlYWRlclwiOntcImtpZFwiOlwiYXJuOmF3czprbXM6dXMtZWFzdC0xOjg5ODA4NDk4NDg3NzprZXkvbXJrLWM3OWM0Y2Y2Yjk3MDQ2NDFhZDhkOTJmNjI0NGM4MzUwXCJ9LFwiaXZcIjpcInFOZlBaQ2ZGcGdlcnhtamNcIixcInByb3RlY3RlZFwiOlwiZXlKaGJHY2lPaUpCVjFOZlJVNURYMU5FUzE5Qk1qVTJJaXdpWlc1aklqb2lRVEkxTmtkRFRTSXNJbnBwY0NJNklrUkZSaUo5XCIsXCJ0YWdcIjpcIm5LM2JfZVpzYmdSWHRJRm9IX1QzZHdcIn0iLCJleHAiOjE3NTI2MTY0NDEsImlhdCI6MTc1MjYxNTg0MSwiaXNzIjoiaHR0cHM6Ly9pZGVudGl0eS5jaWFtLnJlZmluaXRpdi5jb20vYXBpL2lkZW50aXR5L3N0c19wcm9kIiwicnMxIjoiMDhmMmRkZDc5ZjY5ZTMzYWEwYWU4N2U3ZmQ3YzkyODRhZTk0MGFhMSJ9.OHv7SVLzPgoaxtM4bVNDB8BXiXYKESoLendm-xeYoAmeMYP4k0oDPGCxn3kw-HcS6Xd5Ic8WWIs__H9JOqpr12JULhWBQQWKroYeG7jfNnGQqRfWpAqd5D0yn-q0akU5wO_lnw9q3UHCCjp7G3nD7-t5cfgUW-pocZIS1zmCWfd3GkHLJiQhTTVxgbeehRBKzaNWzN1JIUgTfe7BSh7v_oUMRHhAWP0vg5DvrqfjZVEe36f00w6QRutHZGmWv1mHzl863RfjuuCxecfzdaqgMxdH1AAHSPeSjqZk5wkd_l7aj5kNIOVeKotUrEelA6Z8IcP2zMm8sAr1ftWzGPZEFQ; _dd_s=rum=0&expire=1752616995901"""
# cookie = """hmds-token=AQIC5wM2LY4SfczFtDZhmB5KBm4uGSsAl9aI8wcUZ75rBCM%3D%40AAJTSQACNDAAAlNLABM2MzQ2NDM3OTYzODM4MDg0MjI3AAJTMQACMzc%3D%23; x-login-domain=sts.identity.ciam.refinitiv.net; sessionId=0dc686ff-c582-4c6c-b461-caeb1be54e4e; sharedSessionStartTime=1752677121985; userId=SL1-F2VMEAU; x-cdn-account=eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjhDQkMtSFMyNTYiLCJraWQiOiJmZmM3NDk4YzI2YmM0M2UyODFiMGE4NmZjZmQ4MDc1MiIsInppcCI6IkRFRiIsInR5cCI6IkpXVCIsImN0eSI6IkpXVCJ9.nCdqcoV1lOgbgHjM_5-g20eIObX6b1Ph3l8KITFUB7mcJrOIMG4exUq25OCIvuZQBKgftOuO_9cv1-RwylKLFEyhb8WJRd_yW27jy9jbqwnHXRCBsREGYKfshJwIwg39cVx_TXS7dzDcg_3g0-6mJp-rcbCZ9zkyvAMVTgLU1srLvvx2hoDDnv7ai6Soe-vtIIId7f0jwGGrtiILkXBvp9fC0jd0MrIM9BUQJl9cCje1ToX-6F0N3Hu-CsPFYCJOHV5oa0rGdLBX91iSAFzOltQ1yUf6Fn_VJLcTqSrrMW2hUVfz0bkMb-VGAkd5GXZ4RFsMElmrfSSY-zkiqaNmAg.a7ELxrlqNKGLbSONbTNS9Q.bdR3M5okCAL0fb2tAeSxcF-es1fxozYwN2k4OKjKXiaYpOMRfkfOj-bErTBL3H1oFnpVyfr3YZUWvKtEKhxgpmdS5Vt35KEyNBQFQVlJy0LOnyh63j2V7cKcimzJpHQejWjHorXVbSfKu-p7q1Je4ht_NXzbDWUL1uHwAL4RVMm94qUjOlUSnqjlzb4yN9NFGopyUUpgsNZL8XvU_cNWBHbjMwqHx-FiG_OwCnxci8gRuUhvBnWme3--tziO0RQ83BZgSsieYzQCd5ejgH72xiawcz_oXHChNCQsA5uPZoM1Kyjq7nYWcR3oRHj9nLAv5dJr1TVDBSVIvwiehAlPgEYKqMqsaVT7HxHsqlmZzZRsMdvOwF7C5QKj5WrP6qHTVtbNomz0LI-Yfmr5-iLe_Bbgr5mwS5_4qlONwwleXr1kf0amjL-FQLwTzRbKBRvN3fj1y7o_pvMlgWuzVyaF5Tk-_zpnb2f-lWYe_hA8Hg0gLGmB7TH4IFGTlx-wLqDR2P6G6x6d54rPQQZgpPWbe9IbQMogvWD3d3SLH7yV0bWQDXR__7C_P_7S_oo96aLE4eKrKGzAPxbROARhNAC4d3hoGsrt1afBZRGV4NBjsxcUNGuia3qPMfdNIvRLttSIj3QlDHmV7gsh4hU5jMyyAyL5mkMcp8WaoWquq9oFIzvliStyjqDJ0EX3Zl2Hlbd7A6rUj2BS_59KAzGFIhNOnaSPZ1utnNdwsoZ1-SaykiMHLCfcDWpDFSQayHjSzgP_vRaxGENKMZ-fWvQxajhAsx6sTt1bR1yYzimIMAtfgraLiI_OrAlVTvPlkQL61EXiehM_ClmDKZJRbxD7zvfsldow597OEJTkGKaLRWLENmzxrNMmfkpQtJKDiaQ6GQ_7yjsRu6AtF_SUVE6DnRYYH5GJ6O18qZmVA-ypHJ4QdQRst_1vgGNaRtU4Q6RaHUFYHAda7fv7W-zh5p3hMNhaMi9FF_N9qK0p2ySgNx1BjVTbf4L8RFn7xYqgGTBcqMfx919XoU69C78M0qYQJXXHgRKb8bOrhoVVpJK02rbDWnCJufPweyN4cPmC5cfy3Rsmi_eivQMR3M4y8uRWmPsku1_ebRl4081kklAVRM8_dnYgZX1s-BIMRMDMX1YkUuq3iLh23SjkCicysrgmZUhPQ2kzeUuSUshQ4u0uFL_Uqf5Eh2U0yiyB_4G5OMg5FyVACwLBPpveIJH6QbBS8z48hu8PJux7bFgiRs85l5swxxpAzgRHt2qfee1aDqBrR8Xqk-qDkGbIQhgsTEkBCFTll3TgIz1bbn8yboRp8yEI0nvaRDAhvevZHxqC10zbUIW3U1anEXqSyHfD5AIp32Xa1I-c_wQH-v97z0CYB1l0ammjSPPo0lek8A92QJLVwgJGT8tGhE267i7GqhTyAlFTEQjXsE8KP0WhH3rVLMMrOkLu2os0rjYh3rMhHSEqPvupLAdIFoJw0uxxUusklRYbnAXHz0CBApm0Tw49gTtqS5dspwGGSSIbF7XwrJMoGZSHUrF8e5XrpbcX5S6gC6EOPbfG1t7AWqDwbwbK-8gei1f5hIUm6cNX2FBmTLYAjZkAxOmvdxo-GhdLk3weo13iFwf3Qm4GKXI_ZHpetUiTtpWU4fn-WVHVWLMl4I6gBK7Tqdj52Y4OUXi-mWrGuwT4sdAvVPo9V4TdI7k9u5-L6z-k3oYfkxRPgjxNlHhkn4Oh.n39GJ_aYs5LwQQd9IQiRDg; x-sts-token=eyJhbGciOiJSUzI1NiIsImtpZCI6ImRMdFd2Q0tCSC1NclVyWm9YMXFod2pZQ2t1eDV0V2ZSS2o4ME9vcjdUY28iLCJ0eXAiOiJhdCtqd3QifQ.eyJhdWQiOiIzNDYzYTU5MjEzNDQ0NDI4YmVlNWVhZDg4ZmYxZDkwMDQ0NDFiNjU0IiwiZGF0YSI6IntcImNpcGhlcnRleHRcIjpcIjRCcG5vR0ZPblRrVVZnbTBHQXhob2VqMHVBVnVWTXdpUWJ5MjhiUHpwazlxU3ZyZVlMbldaMVRaSDFjc2dhNUZtVVhTczBXamp6bFZUSmQzNGg3Yk5lbXRQaF9GSS1uOTN1UXNEUHJVTGFGUUtsQ2x0bEJYcUVoU3VrdzJVQy1FYTdfRC15Q0Jsa2Z0dWhOdjZLZktwMnFhaEM5ajdPRTQ0dlE1Y1lXYlV3TjVsTGg2akZIRExQMjBsNDFSZS1tRmgzQ1F5SW1sZnVxdUFZZGQ2SlV0RG9nOUh1dkdKYUxsNWpTUXpEOUEzRlkxOHZSY005X3ZuX1NUVFlnU2xlVlZjeG5xWVMyZzFkSlhJZGNJZi1oWXBBeTl0aWdWQ1RVbDB2dXBOS2ZxQlNnTE54ZFFNQjlYVzl0SVVkWVFjTm5wN1laWGNDbU9DbkxQV3ZSWlNoVm1vcFpPSWdNMlg3MmthMVczOXlHcjI5YkhDUmo2VjNDLVZRM3RDRzJEX09hUGUwRTIzV3R5Vm0xblBoanZXbGNEY2lpNHJMVklpOWw1TjhCSzVlLW9UOXllTjlXM1pZWGx2c3Q0aEpZTlFMRGJEelFPNm1hVXYwTnhuVVRSMVN5NHJzUThQNVJxWldsZElQM1BtWlE3Q2dBQ1Q5VkRoZzY5V1doRWlXTHRqNThkUERhVDNCWUJhNENFRklVNHpJS1pPZmo0aEZUZEtFdEswbkhtS2FvSXRrcXZJT1hNSUFDeE1uYndQMU9ZbHM2azZoZGVTMVZzX0pRcUc2NmxaY0xVb3N6cFA1djBrZjBuOWNXUUIwT2RxcTRBSkVnc2xtVmpXMFoycU9aeE1BM0UwWTd4NmJYVzNKQnlIT3l4Wkc1c25XdkpzWW1EcnppMEZHakwzcTRPdGJFZE5TN29uNnJjc19PZ2phaW9KekFseXIyWnBRXCIsXCJlbmNyeXB0ZWRfa2V5XCI6XCJBUUlCQUhnNDJxMmVwUnBlNFpJWmQ1OHR1YUJFT3FWLTBOLXpmaVpmYUk1UXdTREJvQUc3YkNIMVhoNEJrUmZDX3lYUXZ4c25BQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1EdjVZZi1NaXVpRkRva2pIQWdFUWdEdERPaGhiOGNTT2RjZVZSbThON3BNSUpONDhFWUtPcml5RzZRcXhsNjZaVFMyLU80T2NUVC1BdlhuZEp4WjNxZUxSWkNVU0VKa2p4UURMSUFcIixcImhlYWRlclwiOntcImtpZFwiOlwiYXJuOmF3czprbXM6dXMtZWFzdC0xOjg5ODA4NDk4NDg3NzprZXkvbXJrLWM3OWM0Y2Y2Yjk3MDQ2NDFhZDhkOTJmNjI0NGM4MzUwXCJ9LFwiaXZcIjpcImZ2N0xTR1hLeHdRZlpEa2FcIixcInByb3RlY3RlZFwiOlwiZXlKaGJHY2lPaUpCVjFOZlJVNURYMU5FUzE5Qk1qVTJJaXdpWlc1aklqb2lRVEkxTmtkRFRTSXNJbnBwY0NJNklrUkZSaUo5XCIsXCJ0YWdcIjpcIkIxMlFhVXZQc2dQMHN1OUM5ZzB6TFFcIn0iLCJleHAiOjE3NTI2Nzc4MDksImlhdCI6MTc1MjY3NzIwOSwiaXNzIjoiaHR0cHM6Ly9pZGVudGl0eS5jaWFtLnJlZmluaXRpdi5jb20vYXBpL2lkZW50aXR5L3N0c19wcm9kIiwicnMxIjoiOWZlYmRmNmViZGE3Nzk1ZmIwNjA3ZDMxYjAwODI2NTVjM2UyZGE0YSJ9.TvRw2DbIWD27hZhVfJC0ilpg6eu31E3n-UCNpdCPhvJJmu1YA0qsw0t2A-kbBNNSOSjdIjxEJFC9_-ycaWe36AErA1gHROk2kmKCZvPRHHnK8M4mXVw8qQijZeT8UXuan9sBzOQ_XIFDq0tmSOdyXZPcY9AhSY_Ho3u2HwBK00MDi5IGxGVDjA4VhllvvaM3-Q6Vb_EASPsk_SaHH8JbEujJ-d7GXRu_Vzon1W9P4d30fnFbhZoJCaGP0tiX7Gbg41H3eIrymNuKmuTDR7dXdCvRXatsGPA6uny2awyNRU3-veqqhS1TXDvPCMnptDu1z1DFWGoa8uvevAipVyFfgA; _dd_s=rum=0&expire=1752678177398"""
# cookie = """hmds-token=AQIC5wM2LY4SfczFtDZhmB5KBm4uGSsAl9aI8wcUZ75rBCM%3D%40AAJTSQACNDAAAlNLABM2MzQ2NDM3OTYzODM4MDg0MjI3AAJTMQACMzc%3D%23; x-login-domain=sts.identity.ciam.refinitiv.net; sessionId=0dc686ff-c582-4c6c-b461-caeb1be54e4e; sharedSessionStartTime=1752677121985; userId=SL1-F2VMEAU; x-cdn-account=eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjhDQkMtSFMyNTYiLCJraWQiOiJmZmM3NDk4YzI2YmM0M2UyODFiMGE4NmZjZmQ4MDc1MiIsInppcCI6IkRFRiIsInR5cCI6IkpXVCIsImN0eSI6IkpXVCJ9.SDBz-KWU6UFUgy2428sQcPGY4Ha8dSfjnzA0XOci8f-ceR3z1Myi6g5ta16XMUmGUe_Xu_g1ot3XsI1XJjYJHt1PvYE7WeLdxpmFNBAupoj7wZWXJ57RfeMmveadc307VlLo7ac2FYcE0F1tvGbb9q6ewyHYZ8AyajyYV3C51uvVInFRCo1YE-cJkgBb2qw3UzJgKR7vamtdSY-HWsyVzDAyXxtLUnKTipiI_sri8nDZmoEKlhlWjmHuCLmxGSzIjy20efB1bheRoVkE1Dc56_sVQN43l9DFf3S6Vs8dBak-ghxDAYWFEwxIqeHxEYuk-6FMmnKdwd3-83ld4LWlEQ.ra4FWhNbqsICpXvKE70UJQ.MrEI51d7c77ez1mNFSHBUP0deikPvd4Tzu6K_G-Tjesb-NeekKabStYsilec4_6A1o9Nl7V7dB_AeiudcxXWZPnf9ZcdPCzg1A-HTxhRJlMdq9D6vF_XsSiioP_teDxI61TD2eAzN5_Po9QoK4rFgixayw6bvePfpxycg1y8yDqXf3e1BQeQdLEqSW2dSJwBzQkIq1vpRKgq_tJ8FKU_wZbQN7HgboKNzikewtqNy2Uk8R8vDiba2b8r1suKRKNGS3kDmzRQooEnh24ZSxiFY3UmjfQf86WStr9dcSRWgMohhfqUtvQQeujLvUg7zNB21gC3c19bKdALmH5ZQasbZrlpht9Yofdfx3MjITV82YIYwByKIjADcshFZEkhkH932qwk6rLim15MTHYLhbDgwelufa2ggNbBTmhhNjE4IZ2gZXw5dXJ8KG72lIlyEuiGczGaCEbH_Y-FB4yqRzn7LA-r7hPJMYzBP-VKzE6qVU32USGbaiyIfU-AopE70cM_rHTWJgyza-j0U__LgR1KFRLwVpfevtqkPHPmkR14y1vRlOsZ1zvJmYleeIvZxJtVrSXJwFt6HiUG1puktgcLAlcKjjR8tD1mXpTrjiD6xFHS2vnAFFRc4Qumbzac11rqHR-oxIHyLs0j7u6JTI2WyWHYSQnChdopCZcOmUkjyaPDgfrTxutnxTWqINdGe2JMf-flgBpOTKGFyTxg9LaNJReSc_r6CvssYVpZuXRMDzpoG01zAtJs2OLzSZrSqP566NjTqImzVYE_oQILP8U8h5hqG2EitTMMzMSzeNUSLY8fAoKTyc1o_5a4hPRFPVUjlERDc5b6kwvXdZjhCyvS4AtYf2GQvyvymGN6gG_eyfWCND1TIJIjsJHfBJQRdZFIBDHLKe1dThHVhncIKWX3u9D0yYFS1daFiu6tvPTFnHkR2ND-hMFuqhjGPjT-Tsg5gEy2SorAPnnHPVfB5Tkp3Uevbg3fic8Sd9Af1tcNI2ycYuACmne7f9qlOiOixRHYJ0m5uAXfq6jl-BG6sh6uErT1MmCC0bBJ5wWzJMuBPAMEf3AQSHSOMhZ5cWuX877wC5IgqCgaPgnovIUjrp9Fxt03-of-0-17SAfjzVmu11_s3oMODgReNW6nHq5I6oBcgd-m6MV6J80uY3qI1v3sF9JOuQ9iSlxDoqgT2RcxvAqy6i3fjV3Q-CLqNwTtCU-Nonw_7WeASPJHOQjOESOeohB42-WXmGfvYQKUopzlC2-NGPrOLzUu3CO2owAY3M2JGfudGu8aS1tb7sUvKy5N-yMRbBuUH61bRcpQzwCh--0x_CayHaqENTTszi-isjjACbA3mEGqhgscNT4uqUfepd3wmFO88sPB9OfMv3EVWrLb7nqMftbcQjZvVdvTZvziAhr6rI3gmlu1S0VympKCtHOhoTbVUyBFNzHaEMJeuv64jdJbDeimjwlScXSS0ED2sjXw8E4Y2M7SdcWpud1t4Cyu5kbJ9ixgbETsbRZQIhFZOer8RZ8hyRkWWeU4VQzjjaEKk3t5095fsIvszvNZzbtekpYoY6l3x9n_mGm7-zfsPTJeeuvh6wq-DQJyfVyfoExHlMFgAHhvWjRqXxFgyesujRwYaUUQMd8gXUY86St-psVrnAOaUo1uEGT9Ym262QHnXUxM9MMfftQRvqtkDl9EmBmrTVJY3rtUEjgahiwgmtvO3H8DMhVbqGZozui-.GmMMvDyTdVsJPTbq8VRsSg; x-sts-token=eyJhbGciOiJSUzI1NiIsImtpZCI6ImRMdFd2Q0tCSC1NclVyWm9YMXFod2pZQ2t1eDV0V2ZSS2o4ME9vcjdUY28iLCJ0eXAiOiJhdCtqd3QifQ.eyJhdWQiOiIzNDYzYTU5MjEzNDQ0NDI4YmVlNWVhZDg4ZmYxZDkwMDQ0NDFiNjU0IiwiZGF0YSI6IntcImNpcGhlcnRleHRcIjpcInJUbnIzbUxvTjFJYjdYVjZKQlFvNnN2M01iVFI4UTZLYTRzUGt5LUNBZ0x5TDlNWFo0b3VYUUNhMUJMZG5McE9uaWNtODNpTGVSc1Q2cFlXdkhmT052dUtvdWlGYVZQVXU2MnBDX3FsY2JqdkZPcEdIV3BnYkJ1anQxdlljSlFnXzZVTTgwNDNWR0RtRXZmLUl1OGg2WC1MckREcXFQYWlXVkpRaGRSdTRSdXZRVVNUUXR1VDdBeFdjdnpJVC1yZzdueGZybHByaW5JQ3dYOWUtV2xsZDZxQXB5QkFzZjNSemt4RlpsT1BjQTlFUUFmSm5NS0pWQmZLMnNvRnhKLURoSWdBeWJQTjBza205M2dzSWU5ekJSSXZ0TkpxNVR2Sk9YRWtpY0w4RWExMk1fb1J6bjFBODhZdU9jU04xdURnT2hnV3R0aUdVa2RlUkRUbUNjalc2cE10UWtrUy0zUTJOTU13eW9USFpwWC1FNXlEOW5mYnpQdXJCWnpSZEQwZGxoUVJ4TjlYNjZ6RjJHd3J4bDdEcEhrQVJ0UjVCSGJ3bGtsM0tlV2VTejNqMjBrYWlHQ3RMeFNPOU14VThSTENiV2tncHRybndXTTNjbDA0aXFQQVNyaFBWUU8zR3ZxN05lczlUVjZEWDRIbi1kVEFHZWREd2Z0VUFPU2JncFJUcmF5VF80RHlVanVpN2RRUWw1V3h5QUl1aXU2cXZzOF84bDRNYlFJdEhWRnJYMVNfb3dmVGJwU3dGejVQOFhZVkkzRzE0NW85ZllNdldiNDdYejVJcGhfT3doM0pOTi11Q01RUkw3VGt4THVRQ0RlQUh2Y29zdlg5blBBVlpfeDhQejlSeUxxVnlXVTFjOVFvS0MwRjBDcW9YU1J0eU1yVXZMVkhZRTVoRnJ2dkpjNDYyR2ZCekJsVk1FdzlhejI0WEVrRW5RXCIsXCJlbmNyeXB0ZWRfa2V5XCI6XCJBUUlCQUhnNDJxMmVwUnBlNFpJWmQ1OHR1YUJFT3FWLTBOLXpmaVpmYUk1UXdTREJvQUh0OWM2am1LNFlIMXlMRDBsaGhEOV9BQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU1KLWRLZ3QyeFZqUGdzdzJpQWdFUWdEdFNJUFVIUGtXQ19FMFdhcXRaVHA5c2U4dkx5S3VGOHZBb1RzZC05ZUpNcjcwZy1uZDR0Y1JJNGxWNENLVmRYYzNKaDlzeWRUSEtadzNQVlFcIixcImhlYWRlclwiOntcImtpZFwiOlwiYXJuOmF3czprbXM6dXMtZWFzdC0xOjg5ODA4NDk4NDg3NzprZXkvbXJrLWM3OWM0Y2Y2Yjk3MDQ2NDFhZDhkOTJmNjI0NGM4MzUwXCJ9LFwiaXZcIjpcIklOUzFjc19WWlNvVlBqWHRcIixcInByb3RlY3RlZFwiOlwiZXlKaGJHY2lPaUpCVjFOZlJVNURYMU5FUzE5Qk1qVTJJaXdpWlc1aklqb2lRVEkxTmtkRFRTSXNJbnBwY0NJNklrUkZSaUo5XCIsXCJ0YWdcIjpcIjE0akFoN1B4X1QzcVNGVG1SdktodHdcIn0iLCJleHAiOjE3NTI2ODU3OTgsImlhdCI6MTc1MjY4NTE5OCwiaXNzIjoiaHR0cHM6Ly9pZGVudGl0eS5jaWFtLnJlZmluaXRpdi5jb20vYXBpL2lkZW50aXR5L3N0c19wcm9kIiwicnMxIjoiOWZlYmRmNmViZGE3Nzk1ZmIwNjA3ZDMxYjAwODI2NTVjM2UyZGE0YSJ9.Adp4JMS_0XQ2PdfjHG2FWZ20igTjolttwAwIsLI--td_hs4dwNdEJkyOPhpu9OFIQZMDtGXLbFRUvqgkq74TzjdpIUNjEegTQo5vzY9XQ_NDdnAZSLAqMxImlBYBtnWTOjaWAzmKfh5p30RtFMK42GZ50arXoUwuwENaU-n1QoMqaKDgC0vIIZ0qif5VijOcbRgKQJxvNq4xt4iTx9FMt9BnL7SR7F_QNUlqiRxXvoQO6qvCRnapR6zxFGiCNKiwIrMwa7oni2mppYJOMh22_Ke6cDNsfPPXlQAeBPRrhDjh55blrXieu3LB7N0_7S8NXPuOcgtdir6CFQJhPAi5fw; _dd_s=rum=0&expire=1752686112157"""
cookie = """hmds-token=AQIC5wM2LY4SfczFtDZhmB5KBm4uGSsAl9aI8wcUZ75rBCM%3D%40AAJTSQACNDAAAlNLABM2MzQ2NDM3OTYzODM4MDg0MjI3AAJTMQACMzc%3D%23; x-login-domain=sts.identity.ciam.refinitiv.net; sessionId=0dc686ff-c582-4c6c-b461-caeb1be54e4e; sharedSessionStartTime=1752677121985; userId=SL1-F2VMEAU; x-cdn-account=eyJhbGciOiJSU0EtT0FFUCIsImVuYyI6IkExMjhDQkMtSFMyNTYiLCJraWQiOiJmZmM3NDk4YzI2YmM0M2UyODFiMGE4NmZjZmQ4MDc1MiIsInppcCI6IkRFRiIsInR5cCI6IkpXVCIsImN0eSI6IkpXVCJ9.Ber7Wxo1_JLRicg8BAdcmbtYOtAViL6KXdQaXP5sQwsKnzv_r0VwwYaLQdsLd5aggHtQO_-J7fyyNyF33N6o2Tee_Ntr9V4JA-8G9O_M9IQ0-YIeafbdgMb4RtgyT4qq9Ib50eMtGRB2je2KwI81ECgayNIv6Oyn8z0i4nAucNd9mIUT2Lrrw0oaek2C9GMbuKNratSYA8J347adr5iklGZZJYgox4t79tbzyA0e2PxPuuHzecyMnXvouVObYYI3hIPdKfZYOyPJaZKKM5dteTAmGIjHkbz1kqVRrtQJqtFBtUKHiP_5xvsVtCEmici58g-tjWtxNzLx9vbekkld4A.5fZt4P4GzqCBOA7qsN5pfQ.PCKpHP3DZU8O-QvhX986utxfHiz-fhcloCl-xU4QGShvir02lmmQGC2W_dJvb9HZqmlB2QZlaZ_jfVPsUS_8qITROObhkmUlD98Lyweuw3OUzRzE8x-hLA7w_f61cm8ymhZm6EL4d4JzW_oveXqMKEbNBlDDGOgsypok4mya-a17FqZ4Ypmqya2AEMIs7MMDZpWqnEW8Gh6rHUVBhrvOHSCve1MW_szAX7H1N2T_Cbl_9jXGGkA89Zz4kwi_7IaIZ1QBYA30CDgb2KPtidWfDyry6vLi1AI6rA2n-nlwrzYqt_uGHuhmAAiiMsvbWs_SH3FJAOnSuPGfzk7X7Grnrc0eZQG0vsbGcQZvU-XJG6nmxXJG1Sl5p7QBPqsyml08ijJXE0iPI4V7Tmwt7Da61ISC1i306aFsFkTyc-X_EdNrI9aQf4XzFr6xr08q-FVIsk9_5KfREI26CMzsv6V915xDCGaT8jKYd7BGYXRzzFwI768CfGH-HwrZmZmWgpXtF-C8j8GgzfIhf2Km-cjoWr4qM3bydnpVJAPYq1mcFB64ekLy69vz8Qdeaf3MhGGv33UTUYCf_xo-MDHYew_KTUmXECRy5kywIO2mNHRcO2-yqgzCxmTHyYuOkZFfuQgoLztqigeNg0CG2RhrgWr6PUa4fiIfv22UVnZz263pMhavweXr7Do8-yKVwbhaTC3i_qoGZ4gfbEzzcGxaVfw5Uw_NmzBWuCyIKXoaCihrYLdQ_L5mlRtuSHvXO8r50g5q2mTtDRyABQ4Ckhy1rMWUvNXCD0_WMXuRB2KKD79nI-u1_VsdRw49WnztOTq3ESoNEZtHvKseihg406FNdwieeOTgU_3RuwVhP4pBaiPRTAaoDfzOgsGj2R0ZiWPS4PfaOiI3r0NnzZtmR80qA1foLSQyP7FYE0wOReLn4lZtffjBtVCOb_EYOTlvt71uFIQE0E_bpgKIqmVccfUJcYYtlnkoUWa4_QJ2jSKAsMOn1tlrQbrN0fTWsDvUlrcYtU24StheWUl435UXVOzFfLLrMnc3E-OB6sfwegqNetTxM3l8UWvqcBCIdAo8iewk8maDDbHuIF6EvaxE8TJFTixq5S62GhUTbyzepJCcBwv8eCYR-i6YJjigBElMQ9wStoAiAuX5M_Fa1JwzmQrFIeOeu71x0GMmEGxFuBRShWNStTIhrudM7kysbAWpz007g2w9qWviJkxgd0unDxBoozywnsoELplIVKhDkIw2yQ8VkmakbwbNWgWYwDpJaQ_Dd7NR50NZ4J8j4_RQg_LZUTAMMrCsjhf45TyEKm_Bkbr5ZWlYtqTRTWaWziByRXO_BSvm5KzWabvgDh5ynuvixccVEQGziiOqX70e6P7R17fvLwrqdmDphM8V-9j_kh2YEIEgBipLts10e0FQSi_sjZpCJAMDBmYd0hhd76Tz7zjuu9hMIkJ4aXlHOQlizJIpxuhanvmBF1NTqzJlESFCyw59yZMPwbGzDGjudm4BV6SJWCKSqly7ppN3C480YgGsIK6vC1IyGc62mIVTo9e9CwPbYSp1uzEVdEDIm5iNv58jBaEDqvpYdrdI8PKObsuOHLqpSZcKGzYc3R0OVhBC-zwwQm7qAppzZRZrr7HvIMAl2sO06lCPq_CmkPv7zFT071QrJYBJ_s5A5LBGHridvoMFDCrtzEefAVvRMdlMNBPVdGV5JCmzglIACQ_ub1Wg9kCb._5uZ-UDSzLa-ExvIeEc94Q; x-sts-token=eyJhbGciOiJSUzI1NiIsImtpZCI6ImRMdFd2Q0tCSC1NclVyWm9YMXFod2pZQ2t1eDV0V2ZSS2o4ME9vcjdUY28iLCJ0eXAiOiJhdCtqd3QifQ.eyJhdWQiOiIzNDYzYTU5MjEzNDQ0NDI4YmVlNWVhZDg4ZmYxZDkwMDQ0NDFiNjU0IiwiZGF0YSI6IntcImNpcGhlcnRleHRcIjpcIk5xSGxVcnZEVGxNOWdzNkh6dnRBVEE1UDVTVi1ZU2JPcUhvRVU3bmRSWDBuWGJGNmpvUHZ1RkNENTFQQ3lQdGJZQjVQRThuVWxPa093eTNTTGRCa3RBLU9jX0M3NVZlQWRLSVliVUl3RkIxQU1NTlhTNXRqakJaS1lBQmV6YUdRMXprUG0wNHBaVXV1LVdZYkdpSUYyczJMbmQxeHV3Rkl1dzRjSldRcFNQcXN0ZllkM0hsdnhMSUYwdWZSSUJaSHl0XzlTbTdJdW1vZURoTnVIX0luOHpOTXgwaW9aY0hEa3BGX1QzRVVFLW9STWVsRHdPQUlNcnhUUWV2cDZFaTBnOS1JX3g2Z1hPTXcwdmNtOHpDTzRDdnhFeDByMGRscUJ4Q2pXUlZ3c3FjbExoMTUweFFhN0NIN3lGUk5zVG5TUnhfR2NlRm0xSGpFTERVajFSZGpoWnh0RDJQR2FyNU9iaUdpVkVoRVBLODBvbHdlWU1BanRPY0FlWVdET0w4azVvcFVvaTA0R2ZYRHVPbFVsTFlmUnNTNlRLb2dLc1RnZzRNZjB2MV9sQndyclJuel9jNXRGeFNaa1lHVTFiNV9LLXFENmFXQmpHUU9RUVlJbjJvb2k4WjJ2V0tDcnduUEo1LXBOMFZ6WWZRdUkxVTBRRmJZZ3QzZkozMFR6YldyNmhVaHNMckFtaDdZc2NjWF9pb0tieWFyZlVpcDNidjFnQmNTbnRJWVZUV2VEeURRYjRHcWJpNU1VSlJPbmdYazk2WnZaMzV4dlJRbmpaeTAyRmxoVHc3YW9yZzdKc1BXWDFnMFpNWDZwOWI5aTA1ME5SMV9RcFJLRmY2VlVwYUNra1NsQnpETk9GVHFxUHVDN0pBandNaHZJTzBncXloaDgzcHdsaGZZUFNBZ3VCT0NWTnZXdkhYUHBiZjZNdmM0VDFUbkh3XCIsXCJlbmNyeXB0ZWRfa2V5XCI6XCJBUUlCQUhnNDJxMmVwUnBlNFpJWmQ1OHR1YUJFT3FWLTBOLXpmaVpmYUk1UXdTREJvQUdvTXdEV2pJZy1IMEtHQURkOS1kWW5BQUFBZmpCOEJna3Foa2lHOXcwQkJ3YWdiekJ0QWdFQU1HZ0dDU3FHU0liM0RRRUhBVEFlQmdsZ2hrZ0JaUU1FQVM0d0VRUU0wbV9ucGtDNHhYOW4tYkVGQWdFUWdEdXVBNUlERElTbWJsNEJKNVAySzBwcTYxcUZFZUoxSFRZajR2Y0RYbUN0SnJVWGtXVll1R1UwblQyOHJtczZxSjNMaUlmS0N0S2RlTFM2QlFcIixcImhlYWRlclwiOntcImtpZFwiOlwiYXJuOmF3czprbXM6dXMtZWFzdC0xOjg5ODA4NDk4NDg3NzprZXkvbXJrLWM3OWM0Y2Y2Yjk3MDQ2NDFhZDhkOTJmNjI0NGM4MzUwXCJ9LFwiaXZcIjpcIjd6ZC01Z045T015Y0RhSnlcIixcInByb3RlY3RlZFwiOlwiZXlKaGJHY2lPaUpCVjFOZlJVNURYMU5FUzE5Qk1qVTJJaXdpWlc1aklqb2lRVEkxTmtkRFRTSXNJbnBwY0NJNklrUkZSaUo5XCIsXCJ0YWdcIjpcIkQ2Q2FxZmhRMmpWU0JvZGRSdEpJQndcIn0iLCJleHAiOjE3NTI2OTMyMTcsImlhdCI6MTc1MjY5MjYxNywiaXNzIjoiaHR0cHM6Ly9pZGVudGl0eS5jaWFtLnJlZmluaXRpdi5jb20vYXBpL2lkZW50aXR5L3N0c19wcm9kIiwicnMxIjoiOWZlYmRmNmViZGE3Nzk1ZmIwNjA3ZDMxYjAwODI2NTVjM2UyZGE0YSJ9.BwTRMST2oL6PrQfpOBYvgSS8T4W3NX0CWYOHwhL7zCBPMrHRVapGJiSprJuGQ5d7s75Mcd3JzdQ42sMNjyGwKLDu1YXaajQ4O1xFgyTgvSISEdpk8FYk6NS36JP_6oGA5qsbeJIhSmrF3MlYn3EJMVlBrEng336srwNAdw9sOwWmjj9SkVEloQeQiPnMRALEKxj9pDX-f232VjqHjxCLrgIWYieaVLmzMeHzC_MNoChRpr-PvP0kJg_5-hGXdLGWg7VDBFIA1z3DDpF6iXACsIiQ_oUmAEHg82bBBLwVwJXJlgkrzE10cLWmhnagAR4kJHc3aFFk_7ZvYbpxJBqGKw; _dd_s=rum=0&expire=1752694016808"""

def search_req_body(saleDateFrom, saleDateTo):
    return {
        "saleDateFrom": saleDateFrom.strftime("%Y-%m-%d"),
        "saleDateTo": saleDateTo.strftime("%Y-%m-%d"),
        "evaluationDate": now,
        "either": False,
        "both": True,
        "moodyLong": True,
        "moodyShort": False,
        "sandpLong": True,
        "sandpShort": False,
        "moodyRatingFrom": {
            "value": "UR",
            "label": "UR"
        },
        "moodyRatingTo": {
            "value": "Aaa",
            "label": "Aaa"
        },
        "moodyUnderlyingFrom": {
            "value": "UR",
            "label": "UR"
        },
        "moodyUnderlyingTo": {
            "value": "Aaa",
            "label": "Aaa"
        },
        "sandpRatingFrom": {
            "value": "UR",
            "label": "UR"
        },
        "sandpRatingTo": {
            "value": "AAA",
            "label": "AAA"
        },
        "sandpUnderlyingFrom": {
            "value": "UR",
            "label": "UR"
        },
        "sandpUnderlyingTo": {
            "value": "AAA",
            "label": "AAA"
        },
        "useOfProceeds": {
            "label": "EDUC Charter school",
            "value": "E5"
        },
        "cusips": [],
        "isEntitledToSp": False,
        "isAllBondInsurance": False,
        "isAllIssueType": False
    }

In [ ]:
search_log = []
detail_log = []
# Indexed by daterange
search_log_data = {}
# Indexed by deal_id
detail_log_data = {}

def reset_logs():
    global search_log, detail_log, search_log_data, detail_log_data
    search_log = []
    detail_log = []
    search_log_data = {}
    detail_log_data = {}


In [38]:
# resp = requests.post(search_url, headers={'cookie': cookie}, json=search_req_body(from_date, to_date))
resp = requests.post(search_url, cookies=cookie_dict, json=search_req_body(dt.now() - timedelta(days=30), dt.now()))
resp.text

'{"data":[{"dealId":"0x00106A4007E903D7","state":"CA","issuer":"CALIFORNIA SCH FIN AUTH CHARTER SCH REV","dealAmount":24000000,"datedDate":"2025-07-01","series":"2025-C","corpProject":"River Springs Charter School","leadManager":"N/A","saleDate":"2025-07-01","taxStatus":"US FEDERAL TAX EXEMPT","moody_rating":"N/A / N/A","sand_rating":"BB+ / N/A / N/A","fitch_rating":"N/A / N/A","issue_type":"Bond","offering_type":"Negotiated"},{"dealId":"0x00106AF4742704D9","state":"FL","issuer":"FLORIDA LOC GOVT FIN COMMN EDL FACS REV","dealAmount":198440000,"datedDate":"2025-07-31","series":"2025-A","corpProject":"Bridgeprep Academy Projects","leadManager":"HERBERT J SIMS AND CO INC","saleDate":"2025-07-24","taxStatus":"US FEDERAL TAX EXEMPT","moody_rating":"Ba1 / N/A","sand_rating":"N/A / N/A / N/A","fitch_rating":"N/A / N/A","issue_type":"Bond","offering_type":"Limited"},{"dealId":"0x00106ACB41F50306","state":"OR","issuer":"OREGON ST FACS AUTH CHARTER SCH REV","dealAmount":5805000,"datedDate":"2025

In [22]:
print(resp.text)

{"message":"Route GET:/api/tm3-backend/muni-data-analysis/deal-search/search not found","error":"Not Found","statusCode":404}


In [13]:
auth_url = "https://amers2.identity.ciam.refinitiv.net/auth/UI/Login"
req_body = f"""goto=aHR0cHM6Ly9zdHMuaWRlbnRpdHkuY2lhbS5yZWZpbml0aXYubmV0L29hdXRoMi92MS9hdXRob3JpemU%2FY2xpZW50X2lkPTM0NjNhNTkyMTM0NDQ0MjhiZWU1ZWFkODhmZjFkOTAwNDQ0MWI2NTQmcmVkaXJlY3RfdXJpPWh0dHBzJTNBJTJGJTJGd29ya3NwYWNlLnJlZmluaXRpdi5jb20lMkZhdXRoJTJGY2FsbGJhY2smcmVzcG9uc2VfdHlwZT1jb2RlJnN0YXRlPWNkbi1yZi03NTlmOTZiNjBlYjM0YjM2OGY0YzA3ZDA1NTUzMzViYiZzY29wZT10cmFwaS5laWtvbi5zZXNzaW9uaW5mbyUyMHRyYXBpJTIwdHJhZGluZyUyMGZpbm1lY2hhbmljcyUyMHJkcGFwaSUyMGFwcA%3D%3D&encoded=true&locale=en-US&product=refinitiv&eso=&username=&IDButton=Submit&fs=&epaid=3463a59213444428bee5ead88ff1d9004441b654&realm=&IDToken1=clint%40roundtablefunding.org&IDToken2=Tanya11%21&IDToken3=FALSE&IDToken4=&gx_charset=UTF-8"""
# {
#     'IDToken1': 'clint@roundtablefunding.org',
#     'IDToken2': 'Tanya11!',
#     'IDToken3': 'FALSE',
# }
res = requests.post(auth_url, data=req_body)
display(res)
print(res.text)

<Response [200]>

<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd"><html><head><title>Sign In</title><meta http-equiv="X-UA-Compatible" content="IE=edge"/><meta http-equiv="content-type" content="text/html; charset=utf-8"/><META page=login><link rel="icon" href="../images/favicon.png"/><link rel="shortcut icon" href="../images/favicon.png"/><link rel="stylesheet" type="text/css" href="/auth/css/lunar_commonstyles_spr.css?v=_2.6.3283.01"/><link rel="stylesheet" type="text/css" href="/auth/css/lunarrefinitiv_loginstyles_spr.css?v=_2.6.3283.01"/><script type="text/javascript" src="/auth/js/signin_spr.js?v=_2.6.3283.01"></script><script type="text/javascript" src="/js/as.js?v=_2.6.3283.01"></script><script src="/auth/js/jquery-1.10.1.min.js"></script><script src="/auth/js/common_head.js"></script><meta name="viewport" content="width=device-width, initial-scale=1"><script type="text/javascript">var customMachineValue="";var logoutUrl;if(customMachineValue){logoutUrl=j

In [15]:
import base64
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.print_page_options import PrintOptions
# This is my new favorite library
import chromedriver_autoinstaller
import streamlit as st

from pathlib import Path
import clipboard

chromedriver_autoinstaller.install()

options = Options()
options.binary_location = "/usr/bin/google-chrome"
# make sure it's all in the background
options.add_argument('--headless')
options.add_argument("--disable-gpu")
driver = webdriver.Chrome(service=Service(), options=options)


In [4]:
import pickle
pickle.load(open('/home/zeke/Downloads/log_2025-07-31_15_46_14.pkl', 'rb'))

{'from_date': datetime.datetime(2025, 10, 1, 0, 0),
 'to_date': datetime.datetime(2025, 7, 31, 15, 46, 14, 738166),
 'cookie': {'_dd_s': 'rum=0&expire=1753994381765',
  'x-sts-token': 'eyJhbGciOiJSUzI1NiIsImtpZCI6ImRMdFd2Q0tCSC1NclVyWm9YMXFod2pZQ2t1eDV0V2ZSS2o4ME9vcjdUY28iLCJ0eXAiOiJhdCtqd3QifQ.eyJhdWQiOiIzNDYzYTU5MjEzNDQ0NDI4YmVlNWVhZDg4ZmYxZDkwMDQ0NDFiNjU0IiwiZGF0YSI6IntcImNpcGhlcnRleHRcIjpcInZOM0lybmZxRlpuTl9CUmwtbmh3TG9kOWgyM2FKTlZUcHYzNnRMelYtUW55N1hMdkNMQXhJQ0NqSFNZSmlWa19MNjhIcFZoTmw1ZGxSeFZ5QktrYW4ydmpxbnQwNlVUZ28wZFV4alFQR1llaF8wZFRfNGpmYWpNQS1zb3NTanJNRWhYV0NoSmxGNzFsVzNvWEFqWXZkQjZlOHVCNUNzcy1wOUc4N19vQkFkZ3lnNDQ1ZDRvZTYyMExyN1NMLW5GSUVpcm9tcEhhbVNjZ0NCWDhTOXR4TC1WcHZTTWtWSC1Ma0RlYVFpLUVkT08xOTd6Vjd1TmN3cWQ4cXR6UzlTLVNTTVRvLXRZYlphM0gwZXJrM3NGeFYxR3RGbm9XdjltcUZlLWdUc1haNERwZERXZEtueFNjWjhnSW50X0RiQjF3QkVZODRBQlQ2dlFhb1doTlNWZVg3MWNyRGp6emJjbUtqZnA4VFZzb185YllNWVE1R2ZncmZVREoxNFZub3ZrZWw5VW5TMXNLMFprWVZtbWdfWEo1bUg0NVZrXzNYS3FYdC0zd2FLazVaMUw4R01GcGROWnQyMzNHUWdIVFdqYkw5UDZC

In [33]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = Options()
options.binary_location = "/usr/bin/google-chrome"
# options.add_argument('--headless')
# options.add_argument("--disable-gpu")

driver = webdriver.Chrome(service=Service(), options=options)

username = "clint@roundtablefunding.org"
password = "Tanya11!"
# auth_url = "https://amers2.identity.ciam.refinitiv.net/auth/UI/Login"
auth_url = "https://amers1.identity.ciam.refinitiv.net/auth/UI/Login?epaid=3463a59213444428bee5ead88ff1d9004441b654&goto=https%3A%2F%2Fsts.identity.ciam.refinitiv.net%2Foauth2%2Fv1%2Fauthorize%3Fclient_id%3D3463a59213444428bee5ead88ff1d9004441b654%26redirect_uri%3Dhttps%253A%252F%252Fworkspace.refinitiv.com%252Fauth%252Fcallback%26response_type%3Dcode%26state%3Dcdn-rf-94873aa40fdd411883895893f359bc3a%26scope%3Dtrapi.eikon.sessioninfo%2520trapi%2520trading%2520finmechanics%2520rdpapi%2520app"


In [37]:
cookies = driver.get_cookies()
cookie_dict = {cookie['name']: cookie['value'] for cookie in cookies}
cookies


[{'domain': 'workspace.refinitiv.com',
  'httpOnly': False,
  'name': 'userId',
  'path': '/',
  'sameSite': 'None',
  'secure': True,
  'value': 'SL1-F2VMEAU'},
 {'domain': 'workspace.refinitiv.com',
  'httpOnly': False,
  'name': 'sharedSessionStartTime',
  'path': '/',
  'sameSite': 'None',
  'secure': True,
  'value': '1753991443677'},
 {'domain': 'workspace.refinitiv.com',
  'httpOnly': False,
  'name': 'sessionId',
  'path': '/',
  'sameSite': 'None',
  'secure': True,
  'value': 'd9b4a41d-8c37-4611-a123-a1e5c96b6439'},
 {'domain': 'workspace.refinitiv.com',
  'httpOnly': False,
  'name': 'x-sts-token',
  'path': '/',
  'sameSite': 'None',
  'secure': True,
  'value': 'eyJhbGciOiJSUzI1NiIsImtpZCI6ImRMdFd2Q0tCSC1NclVyWm9YMXFod2pZQ2t1eDV0V2ZSS2o4ME9vcjdUY28iLCJ0eXAiOiJhdCtqd3QifQ.eyJhdWQiOiIzNDYzYTU5MjEzNDQ0NDI4YmVlNWVhZDg4ZmYxZDkwMDQ0NDFiNjU0IiwiZGF0YSI6IntcImNpcGhlcnRleHRcIjpcImk3WFZfOG00cUs2N1R6RWR3WmRPMm5jNzhkUDZzMzVDU1ZJQUtnWlVUdVk2UGRvQUs4MkhfQXdUMXpNSV9fd2VQVlFkenl4dFBWUVZ0U

In [32]:
driver.quit()

In [ ]:

driver.get(auth_url)

# Wait for and fill username
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "#AAA-AS-SI1-SE003"))
).send_keys(username)

# Fill password
driver.find_element(By.CSS_SELECTOR, "#AAA-AS-SI1-SE006").send_keys(password)

# Click login
driver.find_element(By.CSS_SELECTOR, "#AAA-AS-SI1-SE014").click()

# Wait for possible redirect
# WebDriverWait(driver, 10).until(lambda d: d.current_url != auth_url)
WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))  # Wait for any load

# If redirected to page2, click the button
# if driver.current_url.startswith(auth_url_page2):
if "are already signed in" in driver.page_source:
    WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.CSS_SELECTOR, "#AAA-AS-SI2-SE009"))
    ).click()

cookies = driver.get_cookies()
cookie_dict = {cookie['name']: cookie['value'] for cookie in cookies}
cookies

# Optional: Close the browser
# driver.quit()


In [19]:
cookie_dict

{'rtRememberUser': 'QVFJQ3paQUtBTUNENXFaK2JtcVd1clFzVlE9PQ%3D%3D',
 'BIGipServerAAA-AAADASV80-UNV-WWW-80': '!9AwekPm3Nl6mIemlctTI7qJS9r1J0zKOw9ctHTjPjmAwme3R2H18KwbdcwLjtxdM0bCG4kJ7k7U3ZTY=',
 'AuthenticationUILBCookie': 'hdcp-aaadasv02',
 'amlbcookie': '37',
 'iPlanetDirectoryPro': 'AQIC5wM2LY4SfczCWooqzd%2BUIzu27p5QOiO2R2cVF9lFHiE%3D%40AAJTSQACNDAAAlNLABM2MTk1OTM1NzY3OTE0MDY2MzQzAAJTMQACMzc%3D%23',
 'JSESSIONID': 'DF6E0747630ADD112A1367AC688D55C2',
 'custom_epaid': '""',
 'BIGipServerAAA-AAACASV-INT-80': '%2110w552QZvr%2FShCKlctTI7qJS9r1J02aJ3hn9x1i%2BRoY9uSuLURwioWNF%2FpqyEbZZ2I1S%2FbbNpOBurRA%3D'}

In [ ]:
def search(from_date, to_date, log=True, tabs=0):
    print(f'{"\t" * tabs}Searching {from_date} to {to_date}...')
    # resp = requests.post(search_url, headers={'cookie': cookie}, data=search_req_body(from_date, to_date))
    resp = requests.post(search_url, headers={'cookie': cookie}, json=search_req_body(from_date, to_date))
    print(f'{"\t" * tabs}...finished searching with {resp.status_code}')
    search_log.append(resp)
    if resp.status_code == 204:
        return {'data': [], 'overThreshold': False}

    if resp.status_code != 200:
        print('!!! Error searching from', from_date, 'to', to_date, ' !!!')

    j = resp.json()
    # If there's more data, recurse in halfs, and combine the data
    if j['overThreshold']:
        print(f'{"\t" * tabs}Over threshold, recursing...')
        half = from_date + ((to_date - from_date) // 2)
        half1 = search(from_date, half, log=False, tabs=tabs+1)
        half2 = search(half, to_date, log=False, tabs=tabs+1)
        j = {'data': half1['data'] + half2['data'], 'overThreshold': False}
        print(f'{"\t" * tabs}...finished recursing')

    if log:
        search_log_data[(from_date, to_date)] = j

    return j

def details(deal_id):
    print(f'Fetching details for {deal_id}...')
    resp = requests.get(detail_url.format(deal_id=deal_id), headers={'cookie': cookie})
    print(f'...finished fetching details with {resp.status_code}')
    detail_log.append(resp)
    j = resp.json()
    detail_log_data[deal_id] = j
    return j


In [ ]:
def get_schools(from_date, to_date):
    schools = []
    # Search by month
    i = from_date
    while i <= to_date:
        schools += search(i, i+timedelta(days=30))['data']
        i += timedelta(days=30)
    try:
        return pl.DataFrame(schools)
    except:
        return schools

def get_deals(deal_ids):
    school_bonus = []
    coupons = []
    for deal_id in deal_ids:
        d = details(deal_id)['data']

        dd = d['dealDetails']
        for i in dd:
            i['deal_id'] = deal_id
        school_bonus += dd

        cd = d['cusipDetails']
        for i in cd:
            i['deal_id'] = deal_id
        coupons += cd
    try:
        return pl.DataFrame(coupons), pl.DataFrame(school_bonus)
    except:
        return coupons, school_bonus


In [ ]:
schools = get_schools(dt(2016, 12, 21, 0, 0), dt(2025, 7, 16, 0, 0))

In [ ]:
# search_log_data_backup = search_log_data.copy()
data = []
for i in search_log_data.values():
    data += i['data']
schools = pl.DataFrame(data)
# schools.write_csv('schools.csv')

In [ ]:
s = schools['dealId'].to_list()
# 0x0010359A776401AD
coupons, bonus_school_data = get_deals(s[s.index('0x00106AADC00D0218'):])


In [ ]:
deal_details = []
coupon_details = []
for id, d in detail_log_data.items():
    deal = d['data']['dealDetails']
    coupon = d['data']['cusipDetails']
    for i in deal:
        i['deal_id'] = id
    for i in coupon:
        i['deal_id'] = id
    deal_details += deal
    coupon_details += coupon
deals = pl.DataFrame(deal_details)
coupons = pl.DataFrame(coupon_details)


In [ ]:
additional_data = deals.select('first_coupon_date', 'first_call_date', 'security_type', 'bank_qualified', 'sp_rating', 'bond_insurance', 'paying_agent', 'deal_id').rename({'deal_id': 'dealId'})
bonds = schools.unique('dealId').join(additional_data, on='dealId', how='left')

In [ ]:
base = 5.25
coupons = coupons.with_columns(reward=(pl.col('coupon_rate') - base) * pl.col('par_amount'))
target_coupons = coupons.filter(pl.col('coupon_rate') > 6).sort('reward', descending=True)

In [8]:
bonds = pl.read_csv('backup_data/bonds.csv')
coupons = pl.read_csv('backup_data/coupons.csv')
target_coupons = pl.read_csv('backup_data/target_coupons.csv')

In [ ]:
target = target_coupons.rename({'deal_id': 'dealId'}).join(bonds, on='dealId', how='left')

In [11]:
target.write_csv('backup_data/target.csv')

In [48]:
import pickle
file = '''
/home/zeke/Documents/School_Data_Scraper_log_2025-08-02_22:32:40.pkl
'''.strip()
with open(file, 'rb') as f:
    log = pickle.load(f)

log.keys()
print(log['traceback'])

Traceback (most recent call last):
  File "/home/zeke/hello/SchoolData/desktop_scraper.py", line 516, in upload
    .drop_duplicates('dealId')
     ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeke/Software/miniconda3/lib/python3.12/site-packages/pandas/core/frame.py", line 6818, in drop_duplicates
    result = self[-self.duplicated(subset, keep=keep)]
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/zeke/Software/miniconda3/lib/python3.12/site-packages/pandas/core/frame.py", line 6950, in duplicated
    raise KeyError(Index(diff))
KeyError: Index(['dealId'], dtype='object')



In [9]:
import pandas as pd

coupons = pd.read_parquet('coupons_deleteme.parquet')
bonus_school_data = pd.read_parquet('bonus_school_data_deleteme.parquet')
schools = pd.read_parquet('schools_deleteme.parquet')


In [50]:
bonds = pd.read_parquet('bonds_deleteme.parquet')
coupons = pd.read_parquet('coupons_deleteme.parquet')
new_bonds = pd.read_parquet('new_bonds_deleteme.parquet')
new_coupons = pd.read_parquet('new_coupons_deleteme.parquet')
concat_bonds = pd.read_parquet('concat_bonds_deleteme.parquet')
concat_coupons = pd.read_parquet('concat_coupons_deleteme.parquet')

In [70]:
# _bonds = (pd.concat([bonds, new_bonds], ignore_index=True)
#     .drop_duplicates('dealId')
#     .sort_values('saleDate')
#     .reset_index(drop=True)
# )
_coupons = (pd.concat([coupons, new_coupons], ignore_index=True)
    .drop_duplicates('cusip')
    .sort_values('cusip')
    # .reset_index(drop=True)
)
_coupons

,Unnamed: 0,cusip,maturity_date,par_amount,coupon_rate,original_price,original_yield,spread,spread_interpolated,deal_id,evaluated_price,evaluated_yield,spread_to_mmd,spread_to_mmd_interpolated,priceTo
3028,3028.0,00766SAC6,2024-11-01,1315000.0,3.500,100.000,3.500,240.0,240.2,0x001035D3D2FB02DD,NaN,NaN,NaN,NaN,None
3029,3029.0,00766SAD4,2029-11-01,1620000.0,3.875,100.000,3.875,243.5,243.8,0x001035D3D2FB02DD,97.446316,4.535151,206.5,204.5,MATY
3030,3030.0,00766SAE2,2034-11-01,2000000.0,5.000,107.399,4.080,240.0,NaN,0x001035D3D2FB02DD,96.366035,5.503685,241.4,239.4,MATY
3031,3031.0,00766SAF9,2044-11-01,5820000.0,5.000,104.832,4.390,238.0,NaN,0x001035D3D2FB02DD,90.204003,5.853219,153.3,NaN,MATY
4433,4433.0,012430AA2,2046-11-01,17445000.0,4.000,115.149,2.300,67.0,NaN,0x001035057966053A,82.219968,5.416784,99.7,NaN,MATY
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2930,2930.0,98608MAH4,2024-10-15,1045000.0,4.500,100.000,4.500,298.0,335.0,0x001035CC0EDC0239,NaN,NaN,NaN,NaN,None
416,416.0,98608MAJ0,2030-10-15,1065000.0,4.000,106.754,3.200,227.0,227.0,0x0010357B8F2304F6,98.026433,4.424411,187.4,186.4,MATY
417,417.0,98608MAK7,2040-10-15,3845000.0,5.000,110.276,3.750,226.0,NaN,0x0010357B8F2304F6,101.768628,4.615082,71.5,NaN,PARC
418,418.0,98608MAL5,2050-10-15,6275000.0,5.000,108.122,4.000,229.0,NaN,0x0010357B8F2304F6,98.201035,5.127288,57.7,NaN,MATY


,Unnamed: 0,dealId,state,issuer,dealAmount,datedDate,series,corpProject,leadManager,saleDate,...,fitch_rating,issue_type,offering_type,first_coupon_date,first_call_date,security_type,bank_qualified,sp_rating,bond_insurance,paying_agent
0,0,0x0010359DECE801BD,WI,PUBLIC FIN AUTH WIS ED REV,18295000.0,2018-10-31,2018-A,Coral Academy of Science Las Vegas,STIFEL NICOLAUS AND CO INC,2018-10-18,...,N/A / N/A,Bond,Negotiated,07/01/2019,07/01/2028,LOAN AGREEMENT,No,NR / N/A / N/A,None,US BANK NA
1,1,0x001035B63837019D,TX,ARLINGTON HIGHER EDUCATION FINANCE CORPORATION,25380000.0,2017-08-31,2017-B,Uplift Education,ROBERT W BAIRD CO INC,2017-08-22,...,N/A / N/A,Bond,Negotiated,12/01/2017,06/01/2027,LOAN AGREEMENT,No,NR / N/A / N/A,None,BANK OF NEW YORK MELLON TRUST CO NA
2,2,0x0010355C146D0808,TX,CLIFTON TEX HIGHER ED FIN CORP ED REV,61010000.0,2023-06-21,2023-A,Valor Education,ROBERT W BAIRD AND CO INC,2023-06-09,...,N/A / N/A,Bond,Limited,12/15/2023,06/15/2031,LOAN AGREEMENT,No,N/A / N/A / N/A,None,REGIONS BANK
3,3,0x001035D1037D058F,FL,PINELLAS CNTY FLA EDL FACS AUTH REV,315000.0,2021-11-05,2021-B,Athenian Academy Project,HILLTOP SECURITIES INC,2021-11-03,...,N/A / N/A,Bond,Private Placement,06/15/2022,None,LOAN AGREEMENT,No,N/A / N/A / N/A,None,UMB BANK NA
4,4,0x00106A9DF7AE01BF,UT,UTAH ST CHARTER SCH FIN AUTH CHARTER SCH REV,6225000.0,2024-11-12,2024-A,Wasatch Peak Academy Project,RAYMOND JAMES AND ASSOCIATES INC,2024-10-31,...,N/A / N/A,Bond,Negotiated,04/15/2025,10/15/2032,LOAN AGREEMENT,No,AA / N/A / N/A,None,ZIONS BANCORPORATION NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1887,1887,0x00106AF39433042A,MN,COON RAPIDS MINN CHARTER SCH LEASE REV,22695000.0,2025-08-28,2025,Athlos Leadership Academy BrooklynPark,PIPER SANDLER AND CO,2025-07-31,...,N/A / N/A,Bond,Negotiated,12/15/2025,06/15/2035,LEASE - RENT,No,N/A / N/A / N/A,None,None
1888,1888,0x00106A38B60504F5,WI,PUBLIC FIN AUTH WIS ED REV,39505000.0,2025-07-29,2025,Pine Springs Preparatory Academy,None,2025-07-24,...,N/A / N/A,Bond,Private Placement,12/15/2025,07/29/2025,LOAN AGREEMENT,No,N/A / N/A / N/A,None,UMB BANK NA
1889,1889,0x00106A38B60504F5,WI,PUBLIC FIN AUTH WIS ED REV,39505000.0,2025-07-29,2025,Pine Springs Preparatory Academy,None,2025-07-24,...,N/A / N/A,Bond,Private Placement,12/15/2025,07/29/2025,LOAN AGREEMENT,No,N/A / N/A / N/A,None,UMB BANK NA
1890,1890,0x00106ABF8F080446,MN,WOODBURY MINN CHARTER SCH LEASE REV,14225000.0,2025-07-22,2025-A,Woodbury Leadership Academy Project,HILLTOP SECURITIES INC,2025-07-11,...,N/A / N/A,Bond,Negotiated,01/01/2026,07/01/2032,REVENUE,No,BB- / N/A / N/A,None,US BANK TRUST CO NA


In [ ]:
additional_data = (
    bonus_school_data[['first_coupon_date', 'first_call_date', 'security_type', 'bank_qualified', 'sp_rating', 'bond_insurance', 'paying_agent', 'deal_id']]
    .rename(columns={'deal_id': 'dealId'})
    .astype({'dealId': str})
)
# schools['dealId'] = schools['dealId'].astype(str)
# # schools.dtypes
# schools['dealId']
# additional_data['dealId']

# bonds = schools.drop_duplicates('dealId').join(additional_data, on='dealId', how='left')
bonds = pd.merge(schools.drop_duplicates('dealId'), additional_data, on='dealId', how='left')

# additional_data

,dealId,state,issuer,dealAmount,datedDate,series,corpProject,leadManager,saleDate,taxStatus,...,fitch_rating,issue_type,offering_type,first_coupon_date,first_call_date,security_type,bank_qualified,sp_rating,bond_insurance,paying_agent
0,0x00106AF39433042A,MN,COON RAPIDS MINN CHARTER SCH LEASE REV,22695000,2025-08-28,2025,Athlos Leadership Academy BrooklynPark,PIPER SANDLER AND CO,2025-07-31,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Negotiated,12/15/2025,06/15/2035,LEASE - RENT,No,N/A / N/A / N/A,N/A,US BANK TRUST CO NA
1,0x00106AF4742704D9,FL,FLORIDA LOC GOVT FIN COMMN EDL FACS REV,198440000,2025-07-31,2025-A,Bridgeprep Academy Projects,HERBERT J SIMS AND CO INC,2025-07-24,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Limited,12/15/2025,06/15/2032,LOAN AGREEMENT,No,N/A / N/A / N/A,N/A,UMB BANK NA
2,0x00106A38B60504F5,WI,PUBLIC FIN AUTH WIS ED REV,39505000,2025-07-29,2025,Pine Springs Preparatory Academy,N/A,2025-07-24,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Private Placement,12/15/2025,07/29/2025,LOAN AGREEMENT,No,N/A / N/A / N/A,N/A,UMB BANK NA
3,0x00106A38B60504F5,WI,PUBLIC FIN AUTH WIS ED REV,39505000,2025-07-29,2025,Pine Springs Preparatory Academy,N/A,2025-07-24,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Private Placement,12/15/2025,07/29/2025,LOAN AGREEMENT,No,N/A / N/A / N/A,N/A,UMB BANK NA
4,0x00106A38B60504F5,WI,PUBLIC FIN AUTH WIS ED REV,39505000,2025-07-29,2025,Pine Springs Preparatory Academy,ZIEGLER CAPITAL MARKETS,2025-07-24,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Private Placement,12/15/2025,07/29/2025,LOAN AGREEMENT,No,N/A / N/A / N/A,N/A,UMB BANK NA
5,0x00106A38B60504F5,WI,PUBLIC FIN AUTH WIS ED REV,39505000,2025-07-29,2025,Pine Springs Preparatory Academy,ZIEGLER CAPITAL MARKETS,2025-07-24,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Private Placement,12/15/2025,07/29/2025,LOAN AGREEMENT,No,N/A / N/A / N/A,N/A,UMB BANK NA
6,0x00106ABE6B1E04B5,AZ,SIERRA INDUSTRIAL DEVELOPMENT AUTHORITY,108975000,2025-07-31,2025,Wake Preparatory Academy,ROBERT W BAIRD AND CO INC,2025-07-22,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Limited,12/15/2025,06/15/2033,LOAN AGREEMENT,No,N/A / N/A / N/A,N/A,UMB BANK NA
7,0x00106AEE612104D0,UT,UTAH ST CHARTER SCH FIN AUTH CHARTER SCH REV,13070000,2025-07-31,2025,Karl G Maeser Preparatory Academy Improvement,HERBERT J SIMS AND CO INC,2025-07-22,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Negotiated,10/15/2025,04/15/2033,LOAN AGREEMENT,No,AA / N/A / N/A,N/A,US BANK TRUST CO NA
8,0x00106A89AA33046E,MI,WATERFORD CHARTER TWP MICH,9945000,2025-08-20,2025,N/A,RAYMOND JAMES AND ASSOCIATES INC,2025-07-31,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Competitive,05/01/2026,05/01/2035,UNLIMITED GENERAL OBLIGATION,No,AA / N/A / N/A,N/A,US BANK TRUST CO NA


In [89]:
from cryptography.fernet import Fernet
import base64
import hashlib

def password_to_key(password: str) -> bytes:
    return base64.urlsafe_b64encode(hashlib.sha256(password.encode()).digest())

def encrypt_file(filepath: str, password: str, output_path: str):
    key = password_to_key(password)
    fernet = Fernet(key)

    with open(filepath, 'rb') as f:
        data = f.read()
    encrypted = fernet.encrypt(data)

    with open(output_path, 'wb') as f:
        f.write(encrypted)

# Example usage
encrypt_file('backup_data/bonds.csv', 'round_table_rocks', 'backup_data/bonds_encrypted.bin')
encrypt_file('backup_data/coupons.csv', 'round_table_rocks', 'backup_data/coupons_encrypted.bin')


In [80]:
from io import BytesIO
from cryptography.fernet import Fernet
import base64
import hashlib

def password_to_key(password: str) -> bytes:
    return base64.urlsafe_b64encode(hashlib.sha256(password.encode()).digest())
def decrypt_file_to_df(encrypted_path: str, password: str):
    key = password_to_key(password)
    fernet = Fernet(key)

    with open(encrypted_path, 'rb') as f:
        encrypted = f.read()

    try:
        decrypted = fernet.decrypt(encrypted)
    except Exception as e:
        raise ValueError("Invalid password or corrupted file.") from e

    buffer = BytesIO(decrypted)

    return pd.read_csv(buffer)


In [96]:
local_bonds = decrypt_file_to_df('backup_data/bonds_encrypted.bin', 'round_table_rocks')
local_coupons = decrypt_file_to_df('backup_data/coupons_encrypted.bin', 'round_table_rocks')

In [90]:
# Removed unnamed columns and rencrypt
local_bonds.drop('Unnamed: 0', axis=1).to_csv('backup_data/bonds.csv', index=False)
local_coupons.drop('Unnamed: 0', axis=1).to_csv('backup_data/coupons.csv', index=False)
encrypt_file('backup_data/bonds.csv', 'round_table_rocks', 'backup_data/bonds_encrypted.bin')
encrypt_file('backup_data/coupons.csv', 'round_table_rocks', 'backup_data/coupons_encrypted.bin')


In [97]:
local_bonds

,dealId,state,issuer,dealAmount,datedDate,series,corpProject,leadManager,saleDate,taxStatus,...,fitch_rating,issue_type,offering_type,first_coupon_date,first_call_date,security_type,bank_qualified,sp_rating,bond_insurance,paying_agent
0,0x00102100450EC8B2,WI,PUBLIC FIN AUTH WIS EDL FAC REV,6360000.0,2015-02-12,2015-A,The Ask Academy Project,DOUGHERTY AND CO LLC,2015-01-30,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Negotiated,08/01/2015,02/01/2025,LOAN AGREEMENT,No,N/A / N/A / N/A,NaN,WELLS FARGO BANK NA
1,0x00102100450EC3B9,DE,DELAWARE ST ECONOMIC DEV AUTH REV,33965000.0,2015-02-18,2015-A,Odyssey Charter School Inc Project,ROBERT W BAIRD CO INC,2015-02-02,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Negotiated,09/01/2015,03/01/2025,LOAN AGREEMENT,No,N/A / N/A / N/A,NaN,ZIONS FIRST NATIONAL BANK
2,0x0010210048AA4794,CO,COLORADO EDL & CULTURAL FACS AUTH REV,13315000.0,2015-02-17,2015-A,The Classical Academy Project,DA DAVIDSON AND CO,2015-02-10,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Negotiated,06/01/2015,12/01/2024,LOAN AGREEMENT,No,NR / N/A / N/A,NaN,WELLS FARGO BANK NA
3,0x001021004DFBBF38,TX,ARLINGTON HIGHER EDUCATION FINANCE CORPORATION,470000.0,2015-03-12,2015-B,Odyssey Academy Inc,OPPENHEIMER AND CO INC,2015-02-18,US FEDERAL TAXABLE,...,N/A / N/A,Bond,Negotiated,08/15/2015,NaN,LOAN AGREEMENT,No,NR / N/A / N/A,NaN,BANK OF NEW YORK MELLON TRUST CO NA
4,0x001021004D9DE504,TX,ARLINGTON HIGHER EDUCATION FINANCE CORPORATION,11775000.0,2015-03-12,2015-A,Odyssey Academy Inc,OPPENHEIMER AND CO INC,2015-02-18,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Negotiated,08/15/2015,02/15/2025,LOAN AGREEMENT,No,NR / N/A / N/A,NaN,BANK OF NEW YORK MELLON TRUST CO NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1886,0x00106AEE612104D0,UT,UTAH ST CHARTER SCH FIN AUTH CHARTER SCH REV,13070000.0,2025-07-31,2025,Karl G Maeser Preparatory Academy Improvement,HERBERT J SIMS AND CO INC,2025-07-22,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Negotiated,10/15/2025,04/15/2033,LOAN AGREEMENT,No,AA / N/A / N/A,NaN,US BANK TRUST CO NA
1887,0x00106AF4742704D9,FL,FLORIDA LOC GOVT FIN COMMN EDL FACS REV,198440000.0,2025-07-31,2025-A,Bridgeprep Academy Projects,HERBERT J SIMS AND CO INC,2025-07-24,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Limited,12/15/2025,06/15/2032,LOAN AGREEMENT,No,N/A / N/A / N/A,NaN,UMB BANK NA
1888,0x00106A38B60504F5,WI,PUBLIC FIN AUTH WIS ED REV,39505000.0,2025-07-29,2025,Pine Springs Preparatory Academy,NaN,2025-07-24,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Private Placement,12/15/2025,07/29/2025,LOAN AGREEMENT,No,N/A / N/A / N/A,NaN,UMB BANK NA
1889,0x00106AF39433042A,MN,COON RAPIDS MINN CHARTER SCH LEASE REV,22695000.0,2025-08-28,2025,Athlos Leadership Academy BrooklynPark,PIPER SANDLER AND CO,2025-07-31,US FEDERAL TAX EXEMPT,...,N/A / N/A,Bond,Negotiated,12/15/2025,06/15/2035,LEASE - RENT,No,N/A / N/A / N/A,NaN,NaN


In [5]:
%pip install gitpython

Note: you may need to restart the kernel to use updated packages.


''

In [1]:
%pip install argon2

  Preparing metadata (setup.py) ... done
  Created wheel for argon2: filename=argon2-0.1.10-cp312-cp312-linux_x86_64.whl size=19170 sha256=bb7b6223aa87ab8243b57c8d9ae14e7e786684a82d8dba87465cd8be418645a8
  Stored in directory: /home/zeke/.cache/pip/wheels/ba/b0/3b/258f66c0f6f0d76269a770d7daf323f2b3c8df3e841037cce8
Successfully built argon2
Note: you may need to restart the kernel to use updated packages.


In [4]:
from argon2 import PasswordHasher

ph = PasswordHasher()

# When user sets password:
password = "user-password"
ph.hash(password)
# Save stored_hash somewhere

# Later, when user enters password:
# try:
#     ph.verify(stored_hash, "user-password")
#     print("Password correct")
# except:
#     print("Wrong password")


'$argon2id$v=19$m=65536,t=3,p=4$wx3vbueC16m6h3UGB0urHw$YFSm+BKidv75gb3Q9VOXhOnv/CpuqEXQOOGgVj1lTDc'

In [ ]:
import pprp

def hash_password(password: str, salt: bytes = None, iterations: int = 100_000, key_size: int = 32):
    password_bytes = password.encode('utf-8')
    if salt is None:
        salt = os.urandom(16)
    # pprp.pbkdf2(passphrase, salt, key_size)
    # pprp.pbkdf2 uses HMAC‑SHA1, iterations unspecified default (older), so wrap manually:
    dk = pprp.pbkdf2(password_bytes, salt, key_size)  # pprp doesn't expose iteration count, uses default
    return {
        "salt": base64.b64encode(salt).decode(),
        "hash": base64.b64encode(dk).decode(),
        "key_size": key_size
    }

def verify_password(password: str, stored: dict):
    password_bytes = password.encode('utf-8')
    salt = base64.b64decode(stored["salt"])
    expected = base64.b64decode(stored["hash"])
    dk = pprp.pbkdf2(password_bytes, salt, stored["key_size"])
    return dk == expected

hash_password('round_table_rocks')

{'salt': 'yITsoJop3O7QiNnzSdeTSA==',
 'hash': '1KuqXs83yQ6v7vGqYvE2LJG0N4PpvNAYUbk5dHJ8Mes=',
 'key_size': 32}

In [16]:
%pip install pprp
# import pypbkdf2

  Preparing metadata (setup.py) ... done
  Created wheel for pprp: filename=pprp-0.2.7-py3-none-any.whl size=11192 sha256=538e3dde849ada5fa3e97f83b1880ba9c919c658e1b8650f9b3fa89129ee7f68
  Stored in directory: /home/zeke/.cache/pip/wheels/99/83/ed/a09d917b8ae4cd4cc539115e4474bba42aa4425d72f6787e9c
Successfully built pprp
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Extra backup
# bonds.write_csv('bonds.csv')
# target_coupons.write_csv('target_coupons.csv')
# Pickle all the logs
# import pickle
# import json
# with open('detail_log_data.pkl', 'wb') as f:
#     pickle.dump(detail_log_data, f)
# with open('search_log_data.pkl', 'wb') as f:
#     pickle.dump(search_log_data, f)
# with open('detail_log.pkl', 'wb') as f:
#     pickle.dump(detail_log, f)
# with open('search_log.pkl', 'wb') as f:
#     pickle.dump(search_log, f)

# with open('search_log_data.json', 'w') as f:
#     json.dump(search_log_data, f)
# with open('search_log_responses.json', 'w') as f:
#     json.dump(search_log_responses, f)
# with open('detail_log_responses.json', 'w') as f:
#     json.dump(detail_log_responses, f)